In [ ]:
from utils.utils import save_grad_cam, current_time, apply_heatmap, grad_cam
import pandas as pd
from PIL import Image
from torchvision import transforms
from models.efficientnet.enet import enet
from models.vit.vit import vit
import os
import torch
import cv2
import numpy as np

root_path = "./data/AICamp-2023-Skin-Conditions_Dataset"
root_path = "./data/Augmented_Skin_Conditions_Kaggle"
train_csv = 'train.csv'

model_name_suffix = 'efficient_'
enet_model='b0'

load_model = './save_weights/0212_1046_efficient_b0.pt'

# Grad-CAM 시각화
df=pd.read_csv(os.path.join(root_path, 'train.csv'))
num_classes = df['label'].nunique()

if load_model.lower().__contains__('vit'):
    model, feature_extractor = vit(num_classes=num_classes)
    model_name = current_time() + "_" + model + ".pt"
elif load_model.lower().__contains__('efficient'):
    model = enet(model_name=enet_model, num_classes=num_classes)
    model_name = current_time() + "_" + model_name_suffix + enet_model + ".pt"

model.load_state_dict(torch.load(load_model))

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f'Using device: {device}')

model = model.to(device)

target_layer = model._blocks[-1]  
os.makedirs(f'img/grad_cam/{model_name.split(".")[0]}', exist_ok=True)

for i in range(20):
    img_path = os.path.join(root_path, df.loc[i, 'image_path'])

    if model_name.lower().__contains__('vit'):
        img_tensor = feature_extractor(images=Image.open(img_path).convert('RGB'), return_tensors="pt").pixel_values.to(device)
    else:
        img_tensor = transforms.ToTensor()(Image.open(img_path).convert('RGB')).to(device)
    
    # Grad-CAM 이미지 생성
    heatmap = grad_cam(model, img_tensor, target_layer)
    original_img = cv2.imread(img_path)
    # original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
    grad_cam_img = apply_heatmap(original_img, heatmap)

    # 원본 이미지와 Grad-CAM 이미지 결합
    combined_img = np.hstack((original_img, grad_cam_img))

    # 결합된 이미지 저장
    combined_output_path = f'img/grad_cam/{model_name.split(".")[0]}/{i}.jpg'
    cv2.imwrite(combined_output_path, combined_img)

Loaded pretrained weights for efficientnet-b0
Using device: mps


/Users/jeongseung-yeon/miniconda3/envs/skin/lib/python3.12/site-packages/torch/nn/modules/module.py:1830: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)
